# Preparación del dataset en PyTorch

Este notebook cubre la carga, organización, partición y preprocesamiento del dataset para PyTorch.

**Alcance de esta implementación:**
- Estructura del dataset y análisis de etiquetas.
- Generación de splits reproducibles con `seed` fijo.
- Implementación de un `Dataset` personalizado.
- Creación de `DataLoader` para train / valid / test.
- Preprocesamiento sin data augmentation.
- Verificación de un batch final.

In [2]:
from pathlib import Path
import pandas as pd
import yaml
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from torchvision.utils import make_grid
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import subprocess

ROOT = Path("..")
DATA_ROOT = ROOT / "data"
PROCESSED_ROOT = DATA_ROOT / "processed" / "dataset_no_background"
SPLIT_CSV = {
    "train": DATA_ROOT / "train.csv",
    "valid": DATA_ROOT / "val.csv",
    "test": DATA_ROOT / "test.csv",
}

BATCH_SIZE = 16
NUM_WORKERS = 2
SEED = 42

plt.rcParams["figure.figsize"] = (12, 8)

## 0. Eliminación de fondo y creación de `data/processed`

Esta celda procesa el dataset original en `data/raw/dataset/`, elimina el fondo blanco de cada imagen y guarda el resultado en `data/processed/dataset_no_background/`.

In [ ]:
import cv2
import shutil
from pathlib import Path
from tqdm import tqdm

RAW_DATASET = DATA_ROOT / "raw"
PROCESSED_DATASET = PROCESSED_ROOT

WHITE_THRESHOLD = 240
MORPH_KERNEL_SIZE = 3
VALID_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tiff", ".webp"}


def remove_white_background(img_bgr: np.ndarray, threshold: int = WHITE_THRESHOLD, kernel_size: int = MORPH_KERNEL_SIZE) -> np.ndarray:
    h, w = img_bgr.shape[:2]
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    white_mask = (np.all(img_rgb >= threshold, axis=2).astype(np.uint8) * 255)

    flood_mask = np.zeros((h + 2, w + 2), dtype=np.uint8)
    flood_img = white_mask.copy()
    corners = [(0, 0), (0, w - 1), (h - 1, 0), (h - 1, w - 1)]

    for fy, fx in corners:
        if white_mask[fy, fx] == 255:
            cv2.floodFill(
                flood_img,
                flood_mask,
                (fx, fy),
                128,
                loDiff=10,
                upDiff=10,
                flags=cv2.FLOODFILL_FIXED_RANGE,
            )

    bg_mask = (flood_img == 128).astype(np.uint8) * 255
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    bg_mask = cv2.morphologyEx(bg_mask, cv2.MORPH_CLOSE, kernel)
    bg_mask = cv2.morphologyEx(bg_mask, cv2.MORPH_DILATE, kernel)

    alpha = cv2.bitwise_not(bg_mask)
    alpha = cv2.GaussianBlur(alpha, (3, 3), 0)
    _, alpha = cv2.threshold(alpha, 10, 255, cv2.THRESH_BINARY)

    b, g, r = cv2.split(img_bgr)
    img_bgra = cv2.merge([b, g, r, alpha])
    return img_bgra


def process_dataset(raw_dataset: Path, processed_dataset: Path):
    if not raw_dataset.exists():
        raise FileNotFoundError(f"No existe el dataset raw: {raw_dataset.resolve()}")

    splits = ["train", "valid", "test"]
    total_images = 0

    for split in splits:
        input_images = raw_dataset / split / "images"
        input_labels = raw_dataset / split / "labels"
        output_images = processed_dataset / split / "images"
        output_labels = processed_dataset / split / "labels"

        output_images.mkdir(parents=True, exist_ok=True)
        output_labels.mkdir(parents=True, exist_ok=True)

        image_paths = [
            p for p in sorted(input_images.iterdir())
            if p.is_file() and p.suffix.lower() in VALID_EXTENSIONS
        ]

        print(f"\n[{split.upper()}] {len(image_paths)} imágenes encontradas")

        for img_path in tqdm(image_paths, desc=f"Procesando {split}"):
            img_bgr = cv2.imread(str(img_path), cv2.IMREAD_COLOR)
            if img_bgr is None:
                print(f"No se pudo leer: {img_path.name}")
                continue

            img_bgra = remove_white_background(img_bgr)
            out_img = output_images / f"{img_path.stem}.png"
            cv2.imwrite(str(out_img), img_bgra)

            label_file = input_labels / f"{img_path.stem}.txt"
            if label_file.exists():
                shutil.copy2(label_file, output_labels / label_file.name)
            total_images += 1

    yaml_file = raw_dataset / "data.yaml"
    if yaml_file.exists():
        processed_dataset.mkdir(parents=True, exist_ok=True)
        shutil.copy2(yaml_file, processed_dataset / "data.yaml")

    print("\nProcesamiento completado.")
    print(f"Imágenes procesadas: {total_images}")
    print(f"Salida: {processed_dataset.resolve()}")


if RAW_DATASET.exists():
    process_dataset(RAW_DATASET, PROCESSED_DATASET)
else:
    print(f"No existe la carpeta raw dataset: {RAW_DATASET.resolve()}")

No existe la carpeta raw dataset: C:\Users\matia\OneDrive\Documentos\red\Reconocimiento-Patrones-velas-Japonesas\data\raw\dataset


## 1. Estructura del dataset

El dataset procesado está en `data/processed/dataset_no_background/` y conserva la estructura YOLO original:

- `train/images/`
- `train/labels/`
- `valid/images/`
- `valid/labels/`
- `test/images/`
- `test/labels/`

Las etiquetas son archivos `.txt` con formato YOLO: `class x_center y_center width height` en coordenadas normalizadas.

Para PyTorch implementaremos un dataset personalizado porque la estructura no es compatible con `ImageFolder`.

In [ ]:
print("Processed dataset root:", PROCESSED_ROOT.resolve())
print("Exists:", PROCESSED_ROOT.exists())
print("Split CSV files:")
for split, path in SPLIT_CSV.items():
    print(f"  {split}: {path.relative_to(ROOT)} -> {path.exists()}")

yaml_path = PROCESSED_ROOT / "data.yaml"
if yaml_path.exists():
    with open(yaml_path, "r", encoding="utf-8") as f:
        yaml_data = yaml.safe_load(f)
    print("\nclasses:", yaml_data.get("names"))
    print("nc:", yaml_data.get("nc"))
else:
    print("No se encontró data.yaml en el dataset procesado.")

## 2. Generación de splits reproducibles

El repositorio versiona `data/train.csv`, `data/val.csv` y `data/test.csv` con rutas relativas para garantizar que todos trabajen con el mismo particionado.

Si los archivos de split no existen o se desea regenerarlos, se ejecuta `scripts/create_splits.py`.

In [ ]:
if not all(path.exists() for path in SPLIT_CSV.values()):
    script_path = ROOT / "scripts" / "create_splits.py"
    print("Generando splits reproducibles con:", script_path)
    subprocess.run(["python", str(script_path)], check=True)
else:
    print("Los archivos de split ya existen.")

for split, path in SPLIT_CSV.items():
    df = pd.read_csv(path)
    print(f"{split}: {len(df)} filas")
    assert "image" in df.columns and "label" in df.columns

In [ ]:
from collections import Counter

for split, path in SPLIT_CSV.items():
    df = pd.read_csv(path)
    class_counts = Counter()
    for label_path in df["label"]:
        text = (PROCESSED_ROOT / label_path).read_text().strip()
        if text:
            class_counts[int(text.split()[0])] += 1
    print(f"{split} distribution:", dict(class_counts))

## 3. Dataset personalizado de PyTorch

Usamos un `Dataset` que:
- carga cada imagen desde `data/processed/dataset_no_background/...`
- convierte RGBA a RGB si es necesario
- lee la etiqueta clase desde el archivo `.txt`
- aplica transformaciones de preprocesamiento

In [ ]:
CLASS_NAMES = yaml_data.get("names", [])
IMAGE_SIZE = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

base_transforms = T.Compose([
    T.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


def read_yolo_class(label_path: Path) -> int:
    text = label_path.read_text().strip()
    if not text:
        raise ValueError(f"Etiqueta vacía en {label_path}")
    return int(text.split()[0])


class CandlestickDataset(Dataset):
    def __init__(self, csv_path: Path, dataset_root: Path, transform=None):
        self.df = pd.read_csv(csv_path)
        self.dataset_root = dataset_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image_path = self.dataset_root / row["image"]
        label_path = self.dataset_root / row["label"]

        image = Image.open(image_path)
        label = read_yolo_class(label_path)

        if self.transform is not None:
            image = self.transform(image)

        return image, label


train_dataset = CandlestickDataset(SPLIT_CSV["train"], PROCESSED_ROOT, transform=base_transforms)
valid_dataset = CandlestickDataset(SPLIT_CSV["valid"], PROCESSED_ROOT, transform=base_transforms)
test_dataset = CandlestickDataset(SPLIT_CSV["test"], PROCESSED_ROOT, transform=base_transforms)

print("Train samples:", len(train_dataset))
print("Valid samples:", len(valid_dataset))
print("Test samples:", len(test_dataset))

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

batch = next(iter(train_loader))
images, labels = batch
print("Batch images shape:", images.shape)
print("Batch labels shape:", labels.shape)
print("Image range before denorm: min=", images.min().item(), "max=", images.max().item())

In [ ]:
unnormalize = T.Normalize(
    mean=[-m / s for m, s in zip(IMAGENET_MEAN, IMAGENET_STD)],
    std=[1 / s for s in IMAGENET_STD]
)

sample_images = images[:8]
sample_labels = labels[:8]

image_grid = make_grid(unnormalize(sample_images), nrow=4, padding=2)
image_grid = image_grid.permute(1, 2, 0).numpy()
image_grid = np.clip(image_grid, 0, 1)

plt.figure(figsize=(14, 8))
plt.imshow(image_grid)
plt.axis("off")
plt.title("Batch de imágenes de train desnormalizadas")
plt.show()

print("Etiquetas del batch:", sample_labels.tolist())
print("Clases de ejemplo:", [CLASS_NAMES[l] if l < len(CLASS_NAMES) else str(l) for l in sample_labels.tolist()])